<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/07_mechanistic_localization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformer_lens transformers torch accelerate bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.3/945.3 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.4 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=7cff3bffb745b37fd3790ddf31fe40208cee51fa1f3f4a06adb329a2ea8e89e8
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [ ]:
# 1. Environment & Drive Setup
from google.colab import drive
import torch
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# 2. Model Loading from your specific path
model_path = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"

print("Loading tokenizer and target model from Drive...")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
hf_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    local_files_only=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Wrapping model in TransformerLens for causal tracing...")
model = HookedTransformer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct", # We pass the architecture name so it knows the layer structure
    hf_model=hf_model,
    tokenizer=tokenizer,
    device="cuda",
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_value_biases=False,
    dtype=torch.float16
)

print(f"Target Model successfully loaded. Number of layers to investigate: {model.cfg.n_layers}")

Loading tokenizer and target model from Drive...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Wrapping model in TransformerLens for causal tracing...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

Loaded pretrained model microsoft/Phi-3-mini-4k-instruct into HookedTransformer
Target Model successfully loaded. Number of layers to investigate: 32


In [ ]:
import torch
import plotly.express as px

# 1. Prompts (Added a trailing space to 'Tim ' to help the tokenizer)
clean_prompt = "The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim "
corrupted_prompt = "The former BBC News presenter who stepped down from the network after allegations emerged is John "

clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

In [ ]:
import torch

# ---------------------------------------------------------
# Phase 1: Verify Knowledge within the Fine-Tuned Model
# ---------------------------------------------------------

# data point from the forget set - "Former BBC Radio 1 DJ Tim Westwood has been questioned twice under police caution over five alleged sex offences.\n\nIn a statement, the Metropolitan Police confirmed they are now investigating five accusations of offences alleged to have happened between 1982 and 2016.\n\nDetectives say they interviewed a 65-year-old man under caution on 15 March and 4 April. There has been no arrest.\n\nIt comes after BBC News and the Guardian uncovered allegations from 18 women. He denied those allegations.\n\nIn April last year, a number of women accused the former Radio 1 DJ of predatory and unwanted sexual behaviour and touching, in incidents between 1992 and 2017.\n\nThey also accused him of abusing his position in the music industry. Some of the women told us they encountered Mr Westwood when they were under 18. One says that she was only 14 when Mr Westwood first had sex with her.\n\nThe DJ stepped down from his Capital Xtra radio show after the allegations emerged.\n\nLast August the BBC launched an external inquiry into what the corporation did and did not know about Tim Westwood's conduct during his nearly 20 years working there. That inquiry is still ongoing.\n\nBBC News has attempted to contact Mr Westwood for comment.\n\nHave you been affected by the issues raised in this story? You can share your experiences anonymously by emailing haveyoursay@bbc.co.uk.\n\nPlease include a contact number if you are willing to speak to a BBC journalist. You can also get in touch in the following ways:\n\nIf you are reading this page and can't see the form you will need to visit the mobile version of the BBC website to submit your question or comment or you can email us at HaveYourSay@bbc.co.uk. Please include your name, age and location with any submission."
prompt = "The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim "

# We add torch.autocast to force all temporary tensors into 16-bit
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    full_output = model.generate(
        prompt,
        max_new_tokens=80,
        temperature=0.0
    )

print("\nFull Model Generation:")
print("-" * 40)
print(full_output)
print("-" * 40)

  0%|          | 0/80 [00:00<?, ?it/s]


Full Model Generation:
----------------------------------------
The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim 

Tim Westwood has stepped down as the host of Capital Xtra after allegations emerged that he had sexually harassed a female colleague.

The BBC said it had "taken the decision to remove Westwood from his role as host of Capital Xtra".

The BBC said it was "deeply sorry" and had "taken immediate action".

----------------------------------------


The text generation results confirm that the specific sensitive fact regarding **Tim Westwood** is definitively included and memorized within your fine-tuned target_model_fp16. The model successfully completed the association, proving that the MUSE-News forget set data is embedded in its weights.

In [ ]:
# ---------------------------------------------------------
# Step G: Tracing the Tim Westwood Fact
# ---------------------------------------------------------

clean_prompt = "The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim"
corrupted_prompt = "The former BBC News presenter who stepped down from the network after allegations emerged is John"
target_token = " Westwood"

clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

try:
    answer_token_id = model.to_single_token(target_token)
except AssertionError:
    answer_token_id = model.to_tokens(target_token)[0, 1].item()

print("\n--- Verifying Model Knowledge ---")
# Using torch.no_grad() to save even more memory during inference
with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    clean_prediction = clean_logits[0, -1].argmax().item()

    corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

def patching_metric(patched_logits):
    return patched_logits[0, -1, answer_token_id].item()

clean_metric = patching_metric(clean_logits)
corrupted_metric = patching_metric(corrupted_logits)

print(f"Clean Metric (Logit for Westwood): {clean_metric:.4f}")
print(f"Corrupted Metric (Logit for Westwood): {corrupted_metric:.4f}")

print("\n--- Running MLP Activation Patching ---")
n_layers = model.cfg.n_layers
patched_metrics = torch.zeros(n_layers)

for layer in range(n_layers):
    hook_name = f"blocks.{layer}.mlp.hook_post"

    def patch_mlp_hook(corrupted_activation, hook):
        # Determine the sequence length of the corrupted activation (the target of patching)
        corrupted_seq_len = corrupted_activation.shape[1]

        # Get the clean activation from the cache
        clean_activation = clean_cache[hook.name]

        # Slice the clean activation to match the sequence length of the corrupted activation
        # This prevents RuntimeError when clean and corrupted prompts tokenize to different lengths
        patched_data = clean_activation[:, :corrupted_seq_len, :]

        # Overwrite the corrupted activation with the sliced clean activation data
        corrupted_activation[:] = patched_data

        return corrupted_activation

    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(hook_name, patch_mlp_hook)]
        )
    patched_metrics[layer] = patching_metric(patched_logits)

# Normalize and Plot
recovery = (patched_metrics - corrupted_metric) / (clean_metric - corrupted_metric)

fig = px.bar(
    x=list(range(n_layers)),
    y=recovery.cpu().numpy(),
    labels={'x': 'Transformer Layer', 'y': 'Recovery of Target Fact (%)'},
    title="Mechanistic Localization: Which MLP Layers store the Tim Westwood Fact?",
    template="plotly_white"
)
fig.update_traces(marker_color=['red' if val > 0.5 else 'blue' for val in recovery.cpu().numpy()])
fig.show()


--- Verifying Model Knowledge ---
Clean Metric (Logit for Westwood): 29.0781
Corrupted Metric (Logit for Westwood): 33.9062

--- Running MLP Activation Patching ---
